# Bridge Equations for Nowcasting

This notebook demonstrates **bridge equations** — a classic approach used by central banks
worldwide (e.g., ECB, Federal Reserve, Banco Central do Brasil) to nowcast GDP using
monthly indicators.

Bridge equations "bridge" the gap between monthly indicators and the quarterly GDP by:
1. Aggregating monthly indicators to quarterly frequency
2. Estimating a simple regression of quarterly GDP on the aggregated indicators
3. Projecting missing months via AR(1) to handle partial quarters

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.nowcasting import BridgeEquation

# Add helpers path
sys.path.insert(0, "../utils")
from helpers import load_mixed_freq, load_macro_brazil, simulate_ragged_edge

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Bridge Equation Concept

The idea is straightforward: monthly indicators like industrial production, retail sales,
and confidence surveys contain information about the current state of the economy.

A bridge equation connects these monthly indicators to the quarterly GDP target:

$$GDP_Q = \alpha + \beta_1 \bar{x}_{1,Q} + \beta_2 \bar{x}_{2,Q} + \ldots + \varepsilon_Q$$

Where $\bar{x}_{i,Q}$ is the quarterly average (or sum) of the monthly indicator $x_i$.

The key question for nowcasting: what do we do when not all months of the current
quarter are available? Bridge equations handle this by **projecting** missing months
forward (e.g., using an AR(1) model on the monthly series).

In [ ]:
# Load mixed-frequency dataset
data = load_mixed_freq()
print(f"Dataset: {data.shape[0]} monthly observations, {data.shape[1]} variables")
print(f"Date range: {data.index[0].strftime('%Y-%m')} to {data.index[-1].strftime('%Y-%m')}")

# Show frequency differences
print("\n--- Monthly indicators (available every month) ---")
monthly_cols = ["industrial_production", "retail_sales", "confidence_index"]
for col in monthly_cols:
    n_obs = data[col].notna().sum()
    print(f"  {col}: {n_obs} observations")

print("\n--- Quarterly target (only available every 3 months) ---")
gdp_obs = data["gdp_growth"].dropna()
print(f"  gdp_growth: {len(gdp_obs)} observations (quarters)")

# Visualize the frequency mismatch
fig, ax = plt.subplots(figsize=(14, 5))
for col in monthly_cols:
    ax.plot(data.index, data[col], linewidth=1, alpha=0.7, label=col.replace("_", " ").title())
ax.scatter(gdp_obs.index, gdp_obs.values, color="black", s=60, zorder=5, label="GDP Growth (Q)")
ax.set_title("Monthly Indicators vs Quarterly GDP", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Temporal Aggregation

The first step in building a bridge equation is to aggregate the monthly indicators to
quarterly frequency. Common aggregation methods:

- **Mean**: Simple average of the 3 monthly values (most common for indices)
- **Sum**: Sum of the 3 months (appropriate for flow variables like production)
- **Last**: Last month of the quarter (for stock variables)

After aggregation, we can run a simple OLS regression of GDP on the quarterly indicators.

In [ ]:
# Demonstrate temporal aggregation
monthly_indicators = data[monthly_cols]

# Aggregate to quarterly using different methods
quarterly_mean = monthly_indicators.resample("QS").mean()
quarterly_sum = monthly_indicators.resample("QS").sum()
quarterly_last = monthly_indicators.resample("QS").last()

print("Original monthly data (2023):")
print(data.loc["2023", monthly_cols].to_string())

print("\nQuarterly MEAN aggregation (2023):")
print(quarterly_mean.loc["2023"].to_string())

print("\nQuarterly SUM aggregation (2023):")
print(quarterly_sum.loc["2023"].to_string())

# Visualize aggregation effect
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, monthly_cols):
    ax.plot(data.index, data[col], "b-", alpha=0.5, linewidth=0.8, label="Monthly")
    ax.step(quarterly_mean.index, quarterly_mean[col], "r-", linewidth=2, where="mid", label="Quarterly Mean")
    ax.set_title(col.replace("_", " ").title())
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("Monthly to Quarterly Aggregation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Simple OLS of GDP on quarterly-aggregated indicators
gdp_quarterly = data["gdp_growth"].dropna()
common_idx = gdp_quarterly.index.intersection(quarterly_mean.index)
y = gdp_quarterly.loc[common_idx].values
X = quarterly_mean.loc[common_idx].values

# OLS with intercept
X_c = np.column_stack([np.ones(len(y)), X])
beta = np.linalg.lstsq(X_c, y, rcond=None)[0]
y_hat = X_c @ beta
r2 = 1 - np.sum((y - y_hat) ** 2) / np.sum((y - np.mean(y)) ** 2)

print(f"\nManual OLS R-squared: {r2:.4f}")
print(f"Coefficients: intercept={beta[0]:.4f}, IP={beta[1]:.4f}, RS={beta[2]:.4f}, CI={beta[3]:.4f}")

## 3. Single-Indicator Bridge

We start with the simplest case: a bridge equation using only **industrial production**
as the monthly indicator. This is a common baseline in central bank practice.

In [ ]:
# Single-indicator bridge equation: Industrial Production → GDP
bridge_ip = BridgeEquation(
    target="gdp_growth",
    indicators=["industrial_production"],
    aggregation="mean",
    fill_method="ar1",
)

bridge_ip.fit(data)

# Show model summary
print(bridge_ip.summary())

# Nowcast
fc_ip = bridge_ip.nowcast()
print(f"\nNowcast (IP only): {fc_ip.point[0]:.4f}")
print(f"95% CI: [{fc_ip.lower_95[0]:.4f}, {fc_ip.upper_95[0]:.4f}]")
print(f"R-squared: {bridge_ip.r_squared():.4f}")

# Coefficients
print("\nCoefficients:")
print(bridge_ip.coefficients().to_string(index=False))

## 4. Multi-Indicator Bridge

We can improve the nowcast by combining **multiple indicators**. Each indicator
provides a different signal about the economy, and combining them reduces noise.

We evaluate the marginal contribution of each indicator by comparing R-squared values.

In [ ]:
# Multi-indicator bridge: all three monthly indicators
bridge_all = BridgeEquation(
    target="gdp_growth",
    indicators=["industrial_production", "retail_sales", "confidence_index"],
    aggregation="mean",
    fill_method="ar1",
)

bridge_all.fit(data)
print(bridge_all.summary())

# Nowcast with all indicators
fc_all = bridge_all.nowcast()
print(f"\nNowcast (all indicators): {fc_all.point[0]:.4f}")
print(f"95% CI: [{fc_all.lower_95[0]:.4f}, {fc_all.upper_95[0]:.4f}]")

# Compare marginal contribution of each indicator
print("\n--- Marginal Contribution Analysis ---")
indicator_names = ["industrial_production", "retail_sales", "confidence_index"]
r2_values = {}

for ind in indicator_names:
    bridge_single = BridgeEquation(
        target="gdp_growth",
        indicators=[ind],
        aggregation="mean",
    )
    bridge_single.fit(data)
    r2_values[ind] = bridge_single.r_squared()
    print(f"  {ind:30s}  R² = {r2_values[ind]:.4f}")

r2_values["all_combined"] = bridge_all.r_squared()
print(f"  {'all_combined':30s}  R² = {r2_values['all_combined']:.4f}")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = [n.replace("_", "\n") for n in r2_values.keys()]
values = list(r2_values.values())
colors = ["steelblue"] * len(indicator_names) + ["darkorange"]
ax.bar(names, values, color=colors, edgecolor="black", alpha=0.8)
ax.set_ylabel("R-squared")
ax.set_title("Bridge Equation R² by Indicator Combination", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
for i, v in enumerate(values):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Handling Missing Data

The real power of bridge equations for nowcasting is their ability to handle
**partial quarters** — when only 1 or 2 months of the current quarter are available.

The `fill_method` parameter controls how missing months are projected:
- `ar1`: AR(1) forecast for the remaining months
- `last`: Carry forward the last observed value
- `mean`: Fill with the series mean

As more months become available, the nowcast improves because less data is projected.

In [ ]:
# Demonstrate nowcasting with different amounts of quarterly data available
print("=" * 70)
print("Nowcast with partial quarter data")
print("=" * 70)

partial_results = []

for missing_months in [3, 2, 1, 0]:
    months_available = 3 - missing_months
    label = f"{months_available}/3 months available"

    if missing_months > 0:
        partial_data = simulate_ragged_edge(data, {
            "industrial_production": missing_months,
            "retail_sales": missing_months,
            "confidence_index": missing_months,
        })
    else:
        partial_data = data.copy()

    bridge = BridgeEquation(
        target="gdp_growth",
        indicators=indicator_names,
        aggregation="mean",
        fill_method="ar1",
    )
    bridge.fit(partial_data)
    fc = bridge.nowcast()

    partial_results.append({
        "months_available": months_available,
        "nowcast": fc.point[0],
        "lower_95": fc.lower_95[0],
        "upper_95": fc.upper_95[0],
    })
    print(f"  {label}: nowcast = {fc.point[0]:.4f}  "
          f"95% CI = [{fc.lower_95[0]:.4f}, {fc.upper_95[0]:.4f}]")

# Compare fill methods
print("\n--- Fill Method Comparison (2 missing months) ---")
partial_data_2m = simulate_ragged_edge(data, {
    "industrial_production": 2,
    "retail_sales": 2,
    "confidence_index": 2,
})

for method in ["ar1", "last", "mean"]:
    bridge_m = BridgeEquation(
        target="gdp_growth",
        indicators=indicator_names,
        aggregation="mean",
        fill_method=method,
    )
    bridge_m.fit(partial_data_2m)
    fc_m = bridge_m.nowcast()
    print(f"  fill_method='{method}': nowcast = {fc_m.point[0]:.4f}")

# Visualize nowcast convergence
results_df = pd.DataFrame(partial_results)
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(
    results_df["months_available"],
    results_df["nowcast"],
    yerr=[
        results_df["nowcast"] - results_df["lower_95"],
        results_df["upper_95"] - results_df["nowcast"],
    ],
    fmt="o-",
    capsize=5,
    color="steelblue",
    linewidth=2,
    markersize=8,
)
ax.set_xlabel("Months Available in Current Quarter")
ax.set_ylabel("GDP Nowcast")
ax.set_title("Bridge Equation Nowcast as Quarter Progresses", fontsize=13, fontweight="bold")
ax.set_xticks([0, 1, 2, 3])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 1: Build bridge equation for Brazilian GDP

Load the `macro_brazil.csv` dataset and build a bridge equation to nowcast
Brazilian GDP growth using inflation, interest rate, unemployment, and exchange rate
as monthly indicators. Which indicator has the highest predictive power?

In [ ]:
# TODO: Exercise 1
# Hints:
# 1. Load macro_brazil with load_macro_brazil()
# 2. Create BridgeEquation(target='gdp_growth', indicators=[...], aggregation='mean')
# 3. Compare single-indicator vs multi-indicator R²
# 4. Test different aggregation methods ('mean' vs 'sum')
# 5. Generate a nowcast with partial-quarter data

### Exercise 2: Compare bridge with DFM nowcast accuracy

Run a pseudo real-time exercise comparing the bridge equation and DFM approaches.
Which method performs better at different horizons (3, 2, 1 months before GDP release)?

In [ ]:
# TODO: Exercise 2
# Hints:
# 1. Use the pseudo real-time framework from Notebook 01
# 2. At each evaluation date, fit both Bridge and DFM
# 3. Compute RMSE for each model at each horizon
# 4. Plot a comparison chart
# 5. Discuss: when does each approach have an advantage?